# Stock Price Models — Plain-Language Guide

**Open this file in Jupyter Notebook or Jupyter Lab** to see the equations rendered correctly.

This guide explains all four models in this project:
1. GBM (Geometric Brownian Motion)
2. Merton jump-diffusion
3. Heston–Merton
4. GARCH–Merton

For each model you get: the formula, what every symbol means, random variables and their distributions, and real-world meaning.

Interactive demos: `notebooks/01_gbm.ipynb` … `04_garch_merton.ipynb`.


## How the models build on each other

```
GBM  →  add jumps  →  Merton
                         ↓
              add changing volatility
                         ↓
           Heston–Merton    GARCH–Merton
```

- **GBM**: smooth paths, constant volatility (simplest)
- **Merton**: GBM + sudden jumps (crashes, news)
- **Heston–Merton**: changing volatility (random) + jumps
- **GARCH–Merton**: changing volatility (from past data) + jumps


## Shared ideas

### Stock price $S_t$
- **Meaning:** price at time $t$ (e.g. SPY in dollars)
- **Real world:** what you see on a price chart

### Time symbols
| Symbol | Meaning |
|--------|---------|
| $t$ | current time (years) |
| $T$ | total years to simulate |
| $\Delta t$ | one small time step (e.g. 1/252 = one trading day) |

### Log return
If price goes from $S_{\text{old}}$ to $S_{\text{new}}$:

$$r = \ln\!\left(\frac{S_{\text{new}}}{S_{\text{old}}}\right)$$

**Real world:** if $r = 0.01$, price rose about 1% that step.

### Brownian motion $W_t$
Random continuous wiggling. Each step:

$$\Delta W = W_{t+\Delta t} - W_t = \sqrt{\Delta t}\, Z$$

where $Z \sim N(0,1)$ (standard normal: mean 0, variance 1).

**Real world:** everyday market noise, not a single crash headline.

### Drift $\mu$
Average annual trend of log returns. Example: $\mu = 0.08$ means about 8% upward trend per year.

### Monte Carlo
Simulate many possible future paths → a cloud of futures → used for option pricing.


---
# Model 1 — GBM (Geometric Brownian Motion)

**Demo notebook:** `01_gbm.ipynb`

**Idea:** Price moves smoothly. Volatility stays **constant**. This is the Black–Scholes baseline.


### Main equation (continuous time)

$$dS_t = \mu S_t\, dt + \sigma S_t\, dW_t$$

**In words:** the small change in price = (trend × price × time) + (volatility × price × random shock).

Same thing as a percentage change:

$$\frac{dS_t}{S_t} = \mu\, dt + \sigma\, dW_t$$


### Simulation equation (what the code uses)

$$\ln S_{t+\Delta t} - \ln S_t = \Big(\mu - \tfrac{1}{2}\sigma^2\Big)\Delta t + \sigma\sqrt{\Delta t}\, Z$$

| Symbol | Meaning | Real world |
|--------|---------|------------|
| $\mu$ | drift (per year) | average upward/downward trend |
| $\sigma$ | volatility (per year) | how spread out returns are; VIX 15 ≈ $\sigma=0.15$ |
| $S_0$ | start price | today's price |
| $Z$ | random shock | $N(0,1)$ each step |

**Itô correction** $-\tfrac{1}{2}\sigma^2$: keeps prices positive when we use exponentials.

**Random variable:** $Z \sim N(0,1)$ → daily returns are approximately normal (thin tails).

**What GBM misses:** constant vol, no fat tails, no crash jumps.


---
# Model 2 — Merton Jump-Diffusion

**Demo notebook:** `02_merton.ipynb`

**Idea:** GBM + rare **sudden jumps** (earnings, crises, news).


### Main equation

$$\frac{dS_t}{S_{t-}} = (\mu - \lambda\kappa)\, dt + \sigma\, dW_t + (e^J - 1)\, dN_t$$

| Symbol | Meaning | Real world |
|--------|---------|------------|
| $S_{t-}$ | price just before a jump | — |
| $\mu$ | drift | average trend |
| $\sigma$ | diffusion vol | smooth day-to-day moves |
| $\lambda$ | jump intensity | expected **jumps per year** |
| $N_t$ | jump counter | Poisson process |
| $J$ | log jump size | how big the jump is |
| $\kappa$ | jump compensation | keeps math consistent |

**Jump compensation:**

$$\kappa = E[e^J - 1] = e^{\mu_J + \sigma_J^2/2} - 1$$


### Random variables in Merton

**1. Number of jumps in one step**

$$N_{t+\Delta t} - N_t \sim \text{Poisson}(\lambda \Delta t)$$

| $\lambda$ | Plain English |
|-----------|---------------|
| 0.5 | about 1 jump every 2 years |
| 2.0 | about 2 jumps per year |

Most days: zero jumps.

**2. Jump size (when a jump happens)**

$$J \sim N(\mu_J, \sigma_J^2)$$

Price multiplies by $e^J$.

| Parameter | Meaning | Example |
|-----------|---------|---------|
| $\mu_J$ | average log jump | $\mu_J=-0.05$ → roughly −5% jump |
| $\sigma_J$ | jump uncertainty | big value → jump could be tiny or huge |

**Real world:** fat tails, visible kinks in price paths, crisis days.


---
# Model 3 — Heston–Merton

**Demo notebook:** `03_heston_merton.ipynb`

**Idea:** Volatility **changes over time** (Heston) **and** jumps still happen (Merton).


### Two linked equations

**Variance (volatility squared):**

$$dv_t = \kappa(\theta - v_t)\, dt + \xi\sqrt{v_t}\, dW_t^v$$

**Price:**

$$\frac{dS_t}{S_{t-}} = (\mu - \lambda\kappa_J)\, dt + \sqrt{v_t}\, dW_t^S + (e^J - 1)\, dN_t$$

**Correlation:**

$$\text{Corr}(dW_t^S, dW_t^v) = \rho$$

Instantaneous volatility = $\sqrt{v_t}$ (not $v_t$ itself).


### Heston parameters

| Symbol | Name | Meaning | Real world |
|--------|------|---------|------------|
| $v_t$ | variance | vol squared at time $t$ | market stress level |
| $v_0$ | start variance | vol today | $v_0=0.04$ → 20% vol |
| $\theta$ | long-run variance | vol reverts here | normal long-run vol |
| $\kappa$ | mean reversion speed | how fast vol returns to $\theta$ | low $\kappa$ = vol stays high for months |
| $\xi$ | vol-of-vol | randomness in vol itself | high $\xi$ = vol can spike hard |
| $\rho$ | correlation | price shock vs vol shock | usually **negative** (price down → vol up) |

**One simulation step for variance:**

$$\Delta v \approx \kappa(\theta - v)\Delta t + \xi\sqrt{v}\,\sqrt{\Delta t}\, Z_v$$

Jumps use the same $\lambda, \mu_J, \sigma_J$ as Merton.


---
# Model 4 — GARCH(1,1)–Merton

**Demo notebook:** `04_garch_merton.ipynb`

**Idea:** Today’s volatility depends on **yesterday’s shock** and **yesterday’s vol**. Discrete daily steps.


### Equations

**Return this step:**

$$r_t = \mu\,\Delta t + \sigma_t Z_t + J_t$$

**Variance update:**

$$\sigma_t^2 = \omega + \alpha\,\varepsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2$$

**Shock:**

$$\varepsilon_t = \sigma_t Z_t$$

**Price update:**

$$S_{t+\Delta t} = S_t \exp(r_t)$$


### GARCH parameters

| Symbol | Meaning | Real world |
|--------|---------|------------|
| $\omega$ | baseline variance | floor level of vol |
| $\alpha$ | ARCH term | big move yesterday → higher vol today (**clustering**) |
| $\beta$ | GARCH term | vol memory / persistence |
| $\sigma_0$ | starting vol | initial guess |
| $\mu$ | drift | average trend |

**Stability rule:** $\alpha + \beta < 1$

**Long-run variance:**

$$\bar{\sigma}^2 = \frac{\omega}{1 - \alpha - \beta}$$

**Random variables:** $Z_t \sim N(0,1)$; jumps same Poisson + normal structure as Merton.


---
## Comparison table

| Feature | GBM | Merton | Heston–Merton | GARCH–Merton |
|---------|:---:|:------:|:-------------:|:------------:|
| Constant volatility | Yes | Yes | No | No |
| Volatility clustering | No | No | Yes | Yes |
| Sudden jumps | No | Yes | Yes | Yes |
| Fat tails | No | Yes | Yes | Yes |

---

## Link to your American options project

Models simulate future **stock paths** $S_t$. Combined with strike $K$ and rate $r$:

$$\text{call payoff} = \max(S_t - K, 0)$$

Each day: compare **exercise now** vs **wait** → optimal stopping.

| Regime | Years | Market behavior |
|--------|-------|-----------------|
| Crisis | 2008–2009 | high vol, negative jumps |
| Normal | 2013–2014 | lower vol |
| High vol | 2017–2018 | clustering, elevated vol |

---

## Default notebook slider values

| Model | Settings | Read as |
|-------|----------|---------|
| GBM | $\mu=0.08,\ \sigma=0.20$ | 8% drift, 20% vol |
| Merton | $\lambda=0.5,\ \mu_J=-0.05,\ \sigma_J=0.10$ | ~1 jump / 2 years |
| Heston | $\kappa=2,\ \theta=0.04,\ \xi=0.5,\ \rho=-0.6$ | mean-reverting 20% vol |
| GARCH | $\alpha=0.08,\ \beta=0.90$ | strong vol persistence |
